making sure everything is fine and alligned with sql and power bi

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
customers = pd.read_csv("customers.csv")
products = pd.read_csv("products.csv")
stores = pd.read_csv("stores.csv")
transactions = pd.read_csv("transactions.csv")

In [3]:
print("Customers:", customers.shape)
print("Products:", products.shape)
print("Stores:", stores.shape)
print("Transactions:", transactions.shape)

Customers: (200, 7)
Products: (50, 6)
Stores: (5, 4)
Transactions: (5000, 8)


In [4]:
df = transactions.merge(
    products,
    on="ProductID",
    how="left"
)

In [5]:
df.shape

(5000, 13)

In [7]:
df["Revenue"] = (
    df["Quantity"]
    * df["UnitPrice"]
    * (1 - df["Discount"])
)

df["Cost"] = (
    df["Quantity"]
    * df["CostPrice"]
)

df["Profit"] = (
    df["Revenue"]
    - df["Cost"]
)

df["ProfitMargin"] = (
    df["Profit"]
    / df["Revenue"]
)

In [8]:
print("Total Revenue:", round(df["Revenue"].sum(), 2))
print("Total Cost:", round(df["Cost"].sum(), 2))
print("Total Profit:", round(df["Profit"].sum(), 2))
print("Profit Margin:", round(df["Profit"].sum() / df["Revenue"].sum() * 100, 2), "%")

Total Revenue: 14301903.15
Total Cost: 10475588.48
Total Profit: 3826314.67
Profit Margin: 26.75 %


In [9]:
discount_analysis = (
    df.groupby("Discount")
      .agg(
          Transactions=("TransactionID", "count"),
          UnitsSold=("Quantity", "sum"),
          Revenue=("Revenue", "sum"),
          Profit=("Profit", "sum")
      )
      .reset_index()
)

discount_analysis["ProfitMargin"] = (
    discount_analysis["Profit"] /
    discount_analysis["Revenue"] * 100
)

discount_analysis

,Discount,Transactions,UnitsSold,Revenue,Profit,ProfitMargin
0,0.00,1243,3732,3.846472e+06,1.237900e+06,32.182732
1,0.05,1250,3720,3.651419e+06,1.047339e+06,28.683061
2,0.10,1183,3541,3.360176e+06,8.332826e+05,24.798774
3,0.15,1324,3956,3.443836e+06,7.077935e+05,20.552477


From PowwerBI and the table above we notice that the higher the discount the lower the profit margin but we dont want to generalize that easily.
so we basically are trying to answer the following question:
If two transactions have the same quantity and come from the same category, does a higher discount still tend to be associated with lower profit?

In [11]:
import statsmodels.formula.api as smf

In [12]:
model = smf.ols(
    "Profit ~ Discount + Quantity + C(Category)",
    data=df
).fit()

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                 Profit   R-squared:                       0.349
Model:                            OLS   Adj. R-squared:                  0.348
Method:                 Least Squares   F-statistic:                     669.0
Date:                Sat, 08 Aug 2026   Prob (F-statistic):               0.00
Time:                        14:43:05   Log-Likelihood:                -38856.
No. Observations:                5000   AIC:                         7.772e+04
Df Residuals:                    4995   BIC:                         7.776e+04
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                               coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------
Intercept               

Business Conclusion

The analysis shows a clear negative relationship between discounting and profitability. Profit margin decreases from approximately 32.18% with no discount to 20.55% at a 15% discount.

The regression analysis confirms that this relationship remains statistically significant after controlling for quantity sold and product category. A 5-percentage-point increase in discount is associated with approximately $151.49 lower profit per transaction.

Although higher discount levels may support sales volume, the additional volume in this dataset does not fully offset the reduction in profitability. The business should therefore evaluate discount strategies carefully and focus discounts where the expected increase in sales volume is sufficient to compensate for the lower margin.

These results represent statistical associations and should not be interpreted as causal effects.

Notice that  R-squared is about 0.35 which means that discount quantity and category explain about 35% of the variation that's very logical and reasonable because the combination of them alone isn't supposed to explain everything.